# loading raw files from source azure container to delta tables in bronze 

## call_activity

In [0]:
file_path = "/Volumes/cdl/bronze/landing_files/call_activity_2026_06_15.json"

call_activity_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .json(file_path)
)

display(call_activity_df)

In [0]:
bronze_path = "abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/bronze/call_activity/"

(
    call_activity_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

In [0]:
%sql
DROP TABLE cdl.bronze.call_activity;

CREATE TABLE cdl.bronze.call_activity
USING DELTA
LOCATION 'abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/bronze/call_activity/';

In [0]:
%sql
select * from cdl.bronze.call_activity;

In [0]:
%sql
LIST '/Volumes/cdl/bronze/landing_files/';

In [0]:
%sql
select * from cdl.bronze.call_activity


## Defining a manual schema

In [0]:
%run /Workspace/Users/devesh.ud@gmail.com/pharma_commercial_data_lake/code/config_variables

In [0]:
print(f"{source_base_path}/call_activity_2026_06_15.json")

In [0]:
from pyspark.sql.types import *

call_activity_schema = StructType([
    StructField("call_date", StringType(), True),
    StructField("call_id", StringType(), True),
    StructField("call_status", StringType(), True),
    StructField("call_type", StringType(), True),
    StructField("duration_minutes", LongType(), True)])

df_call = spark.read.format("json")\
    .option("header", True)\
    .option("infer_schema",True)\
    .load(f"{source_base_path}/call_activity_2026_06_15.json")

display(df_call)
# .schema(call_activity_schema)\

## printing columns / schema

In [0]:
df_call.columns

In [0]:
df_call.printSchema

## Selecting specific columns

In [0]:
df_call.select("call_date","call_id","call_type").display()

In [0]:
from pyspark.sql.functions import *

df_call2 = df_call.select(
    col("call_date").alias("call_dt"),
    col("call_id").alias("event_id"),
    (col("duration_minutes")/60).alias("duration_hours"))

df_call2.display()

##Filtering the data

In [0]:
df_filtered = df_call.filter("call_type = 'Virtual'" + " and call_status = 'Completed'")
df_filtered.display()

In [0]:
from pyspark.sql.functions import *

df_filter = df_call.filter((col("call_type") == "Virtual") | (col("call_type") == "In-person"))
display(df_filter)


In [0]:
df_filter2 = df_call.filter((col("duration_minutes") > 30 ))
display(df_filter2)

In [0]:
df_call.count()
df_filter3 = df_call.filter(~(col("duration_minutes")==45))
df_filter3.count()

In [0]:
df_call.filter(col("call_type").isin("Virtual","In-person")).display()


## Group by and aggregates

In [0]:
# df_call.display()

df_call.groupBy(col("call_type")).count().display()

In [0]:
df_call.groupBy(col("call_status")).agg(count(col("call_id")).alias("total_calls")).display()

In [0]:
# df_call.display()
df_call.groupBy(col("call_status"),col("call_type")).agg(sum(col("duration_minutes")).alias("total_duration")).display()

In [0]:
# df_call.display()

df_call.groupBy(col("call_status"))\
    .agg(count(col("call_id")).alias("total_calls"),\
        sum(col("duration_minutes")).alias("total_duration")).display()

##Drop duplicates

In [0]:
df_call.display()
# df_call.dropDuplicates(["call_date","call_status"]).display()


In [0]:
df_call.groupBy(col("call_status")).agg(count(col("call_id")).alias("total_calls")).filter(col("total_calls") >= 2).display()

In [0]:
df_call.select(col("call_status")).distinct().display()

In [0]:
from pyspark.sql.functions import *

df_call.select(col("call_date")).dropDuplicates().display()

## Loading HCP and Rx data in bronze schema

In [0]:
LIST 

In [0]:
df_rx = spark.read.format("json")\
    .option("header", True)\
    .option("infer_schema",True)\
    .load(f"{source_base_path}/rx_transactions_2026_06_15.json")

display(df_rx)

In [0]:
df_hcp = spark.read.format("json")\
    .option("headers",True)\
        .option("infer_schema",True)\
            .load(f"{source_base_path}/hcp_master_2026_06_15.json")

df_hcp.display()

In [0]:
df_call.display()


## joining 2 tables

In [0]:
display(df_call)
display(df_hcp)

In [0]:
df_call_hcp = df_call.alias("c").join(df_hcp.alias("h"), on = "hcp_id", how = "inner")

df_call_hcp.display()

In [0]:
df_ch2 = df_call.alias("call").join(
    df_hcp.alias("hcp"),
    col("call.hcp_id") == col("hcp.hcp_id"),
    "inner"
)

df_ch2.display()

In [0]:
df_ch3 = df_call.alias("call").join(
    df_hcp.alias("hcp"),
    (col("call.hcp_id") == col("hcp.hcp_id")) & (col("call.call_date") <= col("hcp.effective_date")),
    "inner"
).select(col("call.*"), col("hcp.hcp_name"),col("hcp.city"))

df_ch3.display()

##windows functions in spark

In [0]:
sales_data = [
    (1, "Delhi", "Amit", "Laptop", 50000),
    (2, "Delhi", "Priya", "Mouse", 800),
    (3, "Delhi", "Rahul", "Keyboard", 1500),
    (4, "Mumbai", "Sneha", "Laptop", 52000),
    (5, "Mumbai", "Karan", "Mouse", 900),
    (6, "Pune", "Neha", "Monitor", 12000),
    (7, "Pune", "Ravi", "Laptop", 51000),
    (8, "Pune", "Ankit", "Keyboard", 1600)
]

sales_columns = ["order_id", "city", "customer_name", "product", "amount"]

df_sales = spark.createDataFrame(sales_data, sales_columns)

display(df_sales)

In [0]:
from pyspark.sql.window import *
from pyspark.sql.functions import *

window_spec = Window.partitionBy("city").orderBy(col("amount").desc())

df_sales2 = df_sales.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num")==1).select("city","customer_name","amount")

df_sales2.display()

In [0]:
df_sales.display()

In [0]:
# second highest customer for each product

from pyspark.sql.window import *

window_def = Window.partitionBy(col("product")).orderBy(col("amount").desc())

df_sales_product = df_sales.withColumn("rank",rank().over(window_def)).filter(col("rank") == 2).select("product","customer_name")

df_sales_product.display()

In [0]:
window_spec = Window.partitionBy("city")

df_result = df_sales.withColumn(
    "city_avg_amount",
    avg("amount").over(window_spec)
).withColumn(
    "amount_vs_city_avg",
    col("amount") - col("city_avg_amount")
)

display(df_result)

In [0]:
df_sales.display()

In [0]:
rank_def = Window.partitionBy(col("city")).orderBy(col("amount").desc())
recurring_sum = Window.orderBy(col("order_id"))
current_pre_sum = Window.orderBy(col("order_id")).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df_sales3 = df_sales.withColumn("rank", rank().over(rank_def)).withColumn("total_sum", sum("amount").over(recurring_sum))\
.withColumn("%age_sum", round(col("amount")/col("total_sum"), 4)).withColumn("cumulative_amount", sum(col("amount")).over(current_pre_sum))
df_sales3.display()